In [3]:
import pandas as pd
import os
from tqdm import tqdm

input_dir = r"C:\Users\28616\Desktop\Correlation_Results"
filtered_dir = r"C:\Users\28616\Desktop\Filtered_Significance"
os.makedirs(filtered_dir, exist_ok=True)

sample_ids = ["49", "50", "51", "52", "53", "54"]

# Header for the console report
print(f"{'Sample':<10} | {'Method':<10} | {'Passed':<10} | {'Dropped':<10} | {'Dropped %':<10}")
print("-" * 65)

for s_id in sample_ids:
    file_path = os.path.join(input_dir, f"Corr_Results_Sample_{s_id}.csv")
    if not os.path.exists(file_path):
        continue
    
    df = pd.read_csv(file_path)
    total_count = len(df)
    
    # =========================================================================
    # 1. Process Spearman: Filter by Rho > 0.3, Sort by Rho (High to Low)
    # =========================================================================
    spearman_sig = df[df['Spearman_Rho'] > 0.3].copy()
    spearman_sig = spearman_sig.sort_values(by='Spearman_Rho', ascending=False)
    spearman_output = spearman_sig[['Site', 'Gene', 'Spearman_Rho', 'Spearman_P']]
    
    # =========================================================================
    # 2. Process Pearson: Filter by P < 0.05, Sort by R (High to Low)
    # =========================================================================
    pearson_sig = df[df['Pearson_P'] < 0.05].copy()
    # 注意：Pearson的相关系数是 R，这里按 Pearson_R 从高到低降序排列
    pearson_sig = pearson_sig.sort_values(by='Pearson_R', ascending=False)
    pearson_output = pearson_sig[['Site', 'Gene', 'Pearson_R', 'Pearson_P']]
    
    # Save to separate CSVs
    spearman_output.to_csv(os.path.join(filtered_dir, f"Sample_{s_id}_Spearman_Sig.csv"), index=False)
    pearson_output.to_csv(os.path.join(filtered_dir, f"Sample_{s_id}_Pearson_Sig.csv"), index=False)
    
    # Stats Calculation & Printing
    for method, passed_count in [("Spearman", len(spearman_output)), ("Pearson", len(pearson_output))]:
        dropped = total_count - passed_count
        percent = (dropped / total_count) * 100 if total_count > 0 else 0
        print(f"{s_id:<10} | {method:<10} | {passed_count:<10} | {dropped:<10} | {percent:>8.2f}%")

print("-" * 65)
print(f"\nDone! Files in '{filtered_dir}' now only contain their respective statistics.")

Sample     | Method     | Passed     | Dropped    | Dropped % 
-----------------------------------------------------------------
49         | Spearman   | 6382       | 4223       |    39.82%
49         | Pearson    | 1356       | 9249       |    87.21%
50         | Spearman   | 6319       | 4721       |    42.76%
50         | Pearson    | 796        | 10244      |    92.79%
51         | Spearman   | 6514       | 5249       |    44.62%
51         | Pearson    | 600        | 11163      |    94.90%
52         | Spearman   | 6779       | 5149       |    43.17%
52         | Pearson    | 819        | 11109      |    93.13%
53         | Spearman   | 7588       | 3851       |    33.67%
53         | Pearson    | 619        | 10820      |    94.59%
54         | Spearman   | 6655       | 5006       |    42.93%
54         | Pearson    | 637        | 11024      |    94.54%
-----------------------------------------------------------------

Done! Files in 'C:\Users\28616\Desktop\Filtered_Significance

In [4]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns

# --- 路径配置 ---
filtered_dir = r"C:\Users\28616\Desktop\Filtered_Significance"
sample_ids = ["49", "50", "51", "52", "53", "54"]
methods = ["Spearman", "Pearson"]

all_counts = []

print("正在统计染色体分布...")

for s_id in sample_ids:
    for method in methods:
        file_path = os.path.join(filtered_dir, f"Sample_{s_id}_{method}_Sig.csv")
        if not os.path.exists(file_path):
            continue
            
        # 读取刚刚筛选出来的显著位点结果
        df = pd.read_csv(file_path)
        
        if 'Site' in df.columns:
            # 核心修改：以 "_" 分割字符串，取第一部分作为染色体
            # 比如 "9_44048108" -> "9", "X_12345" -> "X"
            df['Chromosome'] = df['Site'].astype(str).str.split('_').str[0]
            
            # 统计当前样本、当前方法下，各个染色体的位点数量
            counts = df['Chromosome'].value_counts().reset_index()
            counts.columns = ['Chromosome', 'Count']
            counts['Sample'] = s_id
            counts['Method'] = method
            
            all_counts.append(counts)

if all_counts:
    # 汇总成一张大表
    summary_df = pd.concat(all_counts, ignore_index=True)
    
    # 保存统计明细表
    summary_out_path = os.path.join(filtered_dir, "Chromosome_Distribution_Summary.csv")
    summary_df.to_csv(summary_out_path, index=False)
    print(f"✅ 统计汇总表已保存至: {summary_out_path}")
    
    # --- 高级排序逻辑 ---
    # 确保 X 轴顺序是常理中的 1, 2, 3... 22, 而不是 1, 10, 11, 2...
    def chr_sort_key(c):
        if str(c).isdigit():
            return (0, int(c))  # 数字排在前面，按数值大小排
        else:
            return (1, str(c))  # 字母 (如 X, Y, MT) 排在后面，按字母表排

    # --- 分别绘制 Spearman 和 Pearson 的分布图 ---
    for method in methods:
        plot_data = summary_df[summary_df['Method'] == method]
        if plot_data.empty:
            continue
            
        plt.figure(figsize=(14, 6))
        
        # 获取并排序当前数据中出现过的所有染色体
        unique_chrs = plot_data['Chromosome'].unique()
        order = sorted(unique_chrs, key=chr_sort_key)
        
        sns.barplot(
            data=plot_data, 
            x='Chromosome', 
            y='Count', 
            hue='Sample', 
            order=order, 
            palette='Set2' # 使用柔和一点的配色
        )
        
        plt.title(f'Chromosome Distribution of Significant Sites ({method})', fontsize=15, fontweight='bold')
        plt.xlabel('Chromosome', fontsize=12)
        plt.ylabel('Number of Significant Sites', fontsize=12)
        
        # 把图例放到图表外侧，避免遮挡数据柱子
        plt.legend(title='Sample', bbox_to_anchor=(1.01, 1), loc='upper left')
        plt.tight_layout()
        
        plot_out_path = os.path.join(filtered_dir, f"Chromosome_Distribution_{method}.png")
        plt.savefig(plot_out_path, dpi=300)
        plt.close()
        print(f"📊 {method} 染色体分布柱状图已保存至: {plot_out_path}")
else:
    print("没有找到符合条件的文件或位点信息，请检查文件路径是否正确。")

正在统计染色体分布...
✅ 统计汇总表已保存至: C:\Users\28616\Desktop\Filtered_Significance\Chromosome_Distribution_Summary.csv
📊 Spearman 染色体分布柱状图已保存至: C:\Users\28616\Desktop\Filtered_Significance\Chromosome_Distribution_Spearman.png
📊 Pearson 染色体分布柱状图已保存至: C:\Users\28616\Desktop\Filtered_Significance\Chromosome_Distribution_Pearson.png
